### 📄 Cell documentation
Refer to [`analysis_first_cell.md`](analysis_first_cell.md) for explanation of the following code.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

data_path = "data/nodhavn.geojson"
nodhavn = gpd.read_file(data_path)

### 📄 Cell documentation
Refer to [`analysis_second_cell.md`](analysis_second_cell.md) for explanation of the following code.

In [ ]:
nodhavn.info()
print()
print(nodhavn.head())
print()
print("CRS:", nodhavn.crs)

### 📄 Cell documentation
Refer to [`analysis_third_cell.md`](analysis_third_cell.md) for explanation of the following code.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
nodhavn.plot(ax=ax, edgecolor="black")
ax.set_title("Nodhavn-området")
ax.set_xlabel("")
ax.set_ylabel("")
plt.axis("equal")
plt.show()

### 📄 Cell documentation
Refer to [`analysis_fourth_cell.md`](analysis_fourth_cell.md) for explanation of the following code.

In [ ]:
print("Columns:", list(nodhavn.columns))

# Try to find the category column (expected: "kategori")
category_col = None
for c in nodhavn.columns:
    if str(c).lower() in ["kategori", "category", "type"]:
        category_col = c
        break

fylke_col = next((c for c in nodhavn.columns if str(c).lower() == "fylke"), None)

if category_col is not None:
    print(f"\nUsing category column: {category_col}")
    print(
        "Unique kategori values:",
        sorted(nodhavn[category_col].dropna().unique().tolist()),
    )
    print("\nCounts by kategori:")
    print(nodhavn[category_col].value_counts(dropna=False).sort_index())
else:
    print("\nNo category-like column found (expected 'kategori').")

if fylke_col is not None:
    print(f"\nTop fylke by number of points (column: {fylke_col}):")
    print(nodhavn[fylke_col].value_counts().head(10))
else:
    print("\nNo 'fylke' column found.")

### 📄 Cell documentation
Refer to [`analysis_fifth_cell.md`](analysis_fifth_cell.md) for explanation of the following code.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

if category_col is None:
    print("No category column available; plotting points in a default color.")
    nodhavn.plot(ax=ax, color="steelblue", markersize=20)
else:
    # Convert to string so geopandas treats it as categorical/discrete values.
    nodhavn_to_plot = nodhavn.copy()
    nodhavn_to_plot[category_col] = nodhavn_to_plot[category_col].astype(str)

    nodhavn_to_plot.plot(
        ax=ax,
        column=category_col,
        categorical=True,
        legend=True,
        cmap="viridis",
        markersize=20,
        legend_kwds={"title": "Kategori"},
    )

ax.set_title("Nødhavn fordelt etter kategori (GeoJSON data)")
ax.set_axis_off()
plt.show()

### 📄 Cell documentation
Refer to [`analysis_sixth_cell.md`](analysis_sixth_cell.md) for explanation of the following code.

In [ ]:
# Project to a metric CRS for area computations (approximate in meters)
nodhavn_metric = nodhavn.to_crs(epsg=3857)

# Convex hull of all points: a simple way to describe the spatial spread
hull = nodhavn_metric.geometry.union_all().convex_hull
area_km2 = hull.area / 1e6

centroid_metric = hull.centroid
centroid_lonlat = (
    gpd.GeoSeries([centroid_metric], crs=nodhavn_metric.crs)
    .to_crs(nodhavn.crs)
    .iloc[0]
)

print(f"Approx. convex hull area: {area_km2:.1f} km^2")
print(
    f"Approx. centroid (lon, lat): ({centroid_lonlat.x:.4f}, {centroid_lonlat.y:.4f})"
)

# Plot: points + convex hull outline + centroid
fig, ax = plt.subplots(figsize=(8, 8))
nodhavn_metric.plot(ax=ax, color="steelblue", markersize=10, alpha=0.6)

gpd.GeoSeries([hull], crs=nodhavn_metric.crs).boundary.plot(
    ax=ax, color="crimson", linewidth=2
)
gpd.GeoSeries([centroid_metric], crs=nodhavn_metric.crs).plot(
    ax=ax, color="black", markersize=30
)

ax.set_title("Geografisk utstrekning (convex hull) for nødhavnspunktene")
ax.set_axis_off()
plt.show()

### Vektoranalyse (buffer, overlay, aggregering)

Her brukes **EPSG:25833** (UTM sone 33N) slik at **meter** (f.eks. 500 m buffer) er konsistent over hele Norge.

- **Buffer:** sirkulære soner 500 m rundt hvert nødhavn-punkt.
- **Overlay:** sammenligning av to lag: *convex hull* (hele punktutvalget) og *forening av alle buffere* — med `intersection`, `difference` og `union`.
- **Romlig aggregering:** `dissolve` av buffere per **kommune** (evt. **fylke** hvis kommune mangler), pluss `spatial join` som alternativ måte å telle punkter per område på.

In [ ]:
# --- Buffer (500 m) og overlay mot convex hull ---
BUFFER_M = 500
CRS_METRIC = "EPSG:25833"

nod_metric = nodhavn.to_crs(CRS_METRIC)
kommune_col = next(
    (c for c in nod_metric.columns if str(c).lower() == "kommune"), None
)

# Ett punkt per rad → buffer i meter
nod_buffers = nod_metric.copy()
nod_buffers["geometry"] = nod_buffers.geometry.buffer(BUFFER_M)

# Ett samlet «dekning»-polygon (alle buffere slått sammen)
buffers_union = nod_buffers.geometry.union_all()

# Convex hull av alle punktene (studieområde)
study_hull = nod_metric.geometry.union_all().convex_hull

# To polygon-lag til overlay (ett trekk hver)
gdf_hull = gpd.GeoDataFrame(
    {"lag": ["convex_hull"]}, geometry=[study_hull], crs=nod_metric.crs
)
gdf_buf_union = gpd.GeoDataFrame(
    {"lag": ["buffers_500m_union"]}, geometry=[buffers_union], crs=nod_metric.crs
)

# Overlay: snitt, union, differanse (hull minus buffer-dekning)
overlay_snitt = gpd.overlay(gdf_hull, gdf_buf_union, how="intersection")
overlay_union = gpd.overlay(gdf_hull, gdf_buf_union, how="union")
overlay_diff = gpd.overlay(gdf_hull, gdf_buf_union, how="difference")

area_km2 = lambda g: float(g.area) / 1e6
print(f"Areal convex hull: {area_km2(study_hull):.1f} km²")
print(f"Areal buffer-union: {area_km2(buffers_union):.1f} km²")
if len(overlay_snitt) > 0:
    print(
        f"Areal snitt (hull ∩ buffer-union): {area_km2(overlay_snitt.geometry.iloc[0]):.1f} km²"
    )
if len(overlay_diff) > 0:
    print(
        f"Areal differanse (hull \\ buffer-union): {area_km2(overlay_diff.geometry.iloc[0]):.1f} km²"
    )

# Kart: punkter, buffere (lett synlig), hull og overlay-resultat
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

nod_metric.plot(ax=axes[0], color="black", markersize=8, alpha=0.85, label="Nødhavn")
nod_buffers.plot(ax=axes[0], alpha=0.25, edgecolor="steelblue", facecolor="cyan", label="500 m buffer")
gpd.GeoSeries([study_hull], crs=nod_metric.crs).boundary.plot(
    ax=axes[0], color="crimson", linewidth=2, label="Convex hull (grense)"
)
axes[0].set_title("Buffer 500 m + convex hull")
axes[0].set_axis_off()
axes[0].legend(loc="lower left", fontsize=8)

overlay_snitt.plot(ax=axes[1], color="green", alpha=0.5, edgecolor="darkgreen", label="Snitt")
overlay_diff.plot(ax=axes[1], color="orange", alpha=0.4, edgecolor="orangered", label="Differanse")
gpd.GeoSeries([study_hull], crs=nod_metric.crs).boundary.plot(
    ax=axes[1], color="crimson", linewidth=1.5, linestyle="--"
)
axes[1].set_title("Overlay: snitt (grønn) og hull \\ buffere (oransje)")
axes[1].set_axis_off()
axes[1].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

# union-overlay kan ha flere rader; vi bruker den sjelden til kart her
print("Overlay union (antall polygon-deler):", len(overlay_union))

In [ ]:
# --- Romlig aggregering: dissolve + telling, og spatial join (sjoin) ---
fylke_col_metric = next(
    (c for c in nod_metric.columns if str(c).lower() == "fylke"), None
)
agg_col = kommune_col or fylke_col_metric

if agg_col is None:
    print("Ingen kolonne 'kommune' eller 'fylke' – hopper over romlig aggregering.")
else:
    label = "kommune" if agg_col == kommune_col else "fylke"

    # Dissolve: ett polygon per enhet (union av alle buffere i enheten)
    by_region = nod_buffers.dissolve(by=agg_col)
    counts = nod_metric.groupby(agg_col).size().rename("antall_nodhavn")
    by_region = by_region.join(counts)
    by_region["areal_buffer_km2"] = by_region.geometry.area / 1e6

    agg_table = by_region[["antall_nodhavn", "areal_buffer_km2"]].sort_values(
        "antall_nodhavn", ascending=False
    )
    print(f"Nødhavn og samlet bufferareal per {label} (topp 15):")
    print(agg_table.head(15).to_string())

    # Romlig join: tell punkter som treffer hvert dissolve-polygon (valgfri kryssjekk)
    polys = by_region.reset_index()[[agg_col, "geometry"]]
    sjoined = gpd.sjoin(
        nod_metric,
        polys,
        how="inner",
        predicate="intersects",
    )
    sj_counts = sjoined.groupby(agg_col).size().rename("via_sjoin")
    print(f"\nTelling via spatial join (intersects), topp 10 {label}:")
    print(sj_counts.sort_values(ascending=False).head(10).to_string())

    # Kart: enheter med farge etter antall punkter
    fig, ax = plt.subplots(figsize=(10, 10))
    by_region.plot(
        ax=ax,
        column="antall_nodhavn",
        legend=True,
        legend_kwds={"label": "Antall nødhavn", "shrink": 0.6},
        cmap="YlOrRd",
        edgecolor="black",
        linewidth=0.3,
        alpha=0.85,
    )
    nod_metric.plot(ax=ax, color="blue", markersize=12, alpha=0.9, label="Nødhavn")
    ax.set_title(f"Romlig aggregering: dissolve av 500 m-buffere per {label}")
    ax.set_axis_off()
    ax.legend(loc="lower left")
    plt.show()